# Parameter Estimation from Process Data

This notebook demonstrates how to estimate model parameters from experimental or
process data using difflow's differentiable framework.

**Topics covered:**
1. Basic parameter estimation with gradient descent
2. Estimating kinetic parameters from steady-state reactor data
3. Multi-parameter estimation (rate constant + activation energy)
4. Dynamic parameter estimation from time-series data
5. Flowsheet-level parameter estimation with recycles
6. Uncertainty quantification (confidence intervals, Bayesian inference)

**Key advantage**: Since difflow is built on JAX, we get automatic gradients
through the entire simulation, enabling efficient optimization even for
complex flowsheets with implicit solvers and recycle loops.

In [ ]:
import os
os.environ['JAX_PLATFORM_NAME'] = 'cpu'

import jax
import jax.numpy as jnp
from jax import random, grad, vmap, jit, hessian
from jax import value_and_grad
import matplotlib.pyplot as plt

# Configure JAX
jax.config.update("jax_enable_x64", True)

# Import difflow
from difflow.streams import make_stream, get_flows, total_flow
from difflow.units import CSTR, CSTRParams
from difflow.dynamic import (
    DynamicCSTR,
    integrate_unit,
    integrate,
)
from difflow import Flowsheet, Unit

print(f"JAX version: {jax.__version__}")

## 1. Basic Parameter Estimation Framework

The general approach for parameter estimation:

1. **Define a model** that takes parameters and predicts outputs
2. **Define a loss function** comparing predictions to measurements
3. **Compute gradients** of the loss with respect to parameters
4. **Optimize** using gradient-based methods

$$\theta^* = \arg\min_\theta \sum_i \left( y_i^{\text{pred}}(\theta) - y_i^{\text{meas}} \right)^2$$

In [ ]:
# Simple example: estimate the decay rate from exponential decay data

# Generate synthetic "experimental" data
key = random.PRNGKey(42)
k_true = 0.3  # True decay rate

t_data = jnp.linspace(0, 10, 20)
y_true = jnp.exp(-k_true * t_data)
noise = random.normal(key, shape=t_data.shape) * 0.02
y_measured = y_true + noise

def model(k, t):
    """Exponential decay model."""
    return jnp.exp(-k * t)

def loss_fn(k):
    """Sum of squared errors."""
    y_pred = model(k, t_data)
    return jnp.sum((y_pred - y_measured)**2)

# Gradient descent
k_est = 0.1  # Initial guess
learning_rate = 0.1
history = []

for i in range(100):
    loss, grad_k = value_and_grad(loss_fn)(k_est)
    k_est = k_est - learning_rate * grad_k
    history.append({'k': float(k_est), 'loss': float(loss)})

print(f"True k: {k_true}")
print(f"Estimated k: {k_est:.4f}")
print(f"Final loss: {history[-1]['loss']:.6f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Data and fit
t_smooth = jnp.linspace(0, 10, 100)
axes[0].scatter(t_data, y_measured, label='Measured data', alpha=0.7)
axes[0].plot(t_smooth, model(k_true, t_smooth), 'g--', label=f'True (k={k_true})', linewidth=2)
axes[0].plot(t_smooth, model(k_est, t_smooth), 'r-', label=f'Estimated (k={k_est:.3f})', linewidth=2)
axes[0].set_xlabel('Time')
axes[0].set_ylabel('y')
axes[0].set_title('Model Fit')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter convergence
axes[1].plot([h['k'] for h in history], 'b-')
axes[1].axhline(k_true, color='g', linestyle='--', label='True value')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('k')
axes[1].set_title('Parameter Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss convergence
axes[2].semilogy([h['loss'] for h in history], 'b-')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_title('Loss Convergence')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. Estimating Kinetic Parameters from Reactor Data

Now let's estimate the rate constant for a CSTR from outlet concentration measurements.

For reaction A → B with rate r = k·C_A:
- Measure outlet concentrations at different flow rates
- Estimate k from the data

In [ ]:
# Define the rate function
def rate_fn(C, T, params):
    """First-order reaction: r = k * C_A"""
    k = params['k']
    return jnp.array([k * C['A']])

# Stoichiometry: A → B
stoich = jnp.array([
    [-1.0],  # A consumed
    [+1.0],  # B produced
])

# Generate synthetic experimental data
k_true = 0.15  # True rate constant (1/s)
V_reactor = 1.0  # m³

# Different inlet flow rates
flow_rates = jnp.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])  # mol/s

# Generate "measured" outlet concentrations
key, subkey = random.split(key)
measured_data = []

for F_A_in in flow_rates:
    # Create CSTR with true parameters
    cstr = CSTR(
        CSTRParams(
            V=V_reactor,
            rate_fn=rate_fn,
            stoich=stoich,
            rate_params={'k': k_true},
            species_order=['A', 'B'],
        ),
        mode="isothermal",
    )
    
    inlet = make_stream({'A': float(F_A_in), 'B': 0.0}, T=350.0, P=101325.0)
    outlet, info = cstr(inlet, T_spec=350.0)
    
    # Add measurement noise
    key, subkey = random.split(key)
    noise = random.normal(subkey) * 0.02 * float(F_A_in)
    F_A_out_measured = float(outlet['F_A']) + noise
    
    measured_data.append({
        'F_A_in': float(F_A_in),
        'F_A_out': F_A_out_measured,
        'conversion': float(info['conversion']['A']),
    })

print("Synthetic experimental data:")
print(f"{'F_A_in':>10} {'F_A_out':>10} {'Conversion':>12}")
for d in measured_data:
    print(f"{d['F_A_in']:>10.2f} {d['F_A_out']:>10.3f} {d['conversion']*100:>11.1f}%")

In [ ]:
def estimate_k_from_cstr_data(measured_data, k_initial, V_reactor, n_iterations=100):
    """Estimate rate constant from CSTR outlet measurements."""
    
    def loss_fn(k):
        """Sum of squared errors for outlet flow predictions."""
        total_loss = 0.0
        
        for data_point in measured_data:
            # Create CSTR with current k estimate
            cstr = CSTR(
                CSTRParams(
                    V=V_reactor,
                    rate_fn=rate_fn,
                    stoich=stoich,
                    rate_params={'k': k},
                    species_order=['A', 'B'],
                ),
                mode="isothermal",
            )
            
            inlet = make_stream(
                {'A': data_point['F_A_in'], 'B': 0.0}, 
                T=350.0, P=101325.0
            )
            outlet, _ = cstr(inlet, T_spec=350.0)
            
            # Squared error
            error = (outlet['F_A'] - data_point['F_A_out'])**2
            total_loss = total_loss + error
        
        return total_loss
    
    # Gradient descent with momentum
    k = k_initial
    velocity = 0.0
    learning_rate = 0.5
    momentum = 0.9
    history = []
    
    for i in range(n_iterations):
        loss, grad_k = value_and_grad(loss_fn)(k)
        velocity = momentum * velocity - learning_rate * grad_k
        k = k + velocity
        k = jnp.maximum(k, 0.001)  # Keep k positive
        
        history.append({'k': float(k), 'loss': float(loss), 'grad': float(grad_k)})
        
        if i % 20 == 0:
            print(f"Iter {i:3d}: k = {k:.4f}, loss = {loss:.6f}")
    
    return k, history

# Run estimation
k_estimated, history = estimate_k_from_cstr_data(
    measured_data, 
    k_initial=0.05,  # Initial guess (far from true value)
    V_reactor=V_reactor,
    n_iterations=100,
)

print(f"\nTrue k: {k_true}")
print(f"Estimated k: {k_estimated:.4f}")
print(f"Relative error: {abs(k_estimated - k_true) / k_true * 100:.2f}%")

In [ ]:
# Compare predictions with estimated vs true parameters
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

F_A_in_range = jnp.linspace(0.3, 3.5, 50)
conversions_true = []
conversions_est = []

for F_A_in in F_A_in_range:
    for k_val, conv_list in [(k_true, conversions_true), (float(k_estimated), conversions_est)]:
        cstr = CSTR(
            CSTRParams(
                V=V_reactor,
                rate_fn=rate_fn,
                stoich=stoich,
                rate_params={'k': k_val},
                species_order=['A', 'B'],
            ),
            mode="isothermal",
        )
        inlet = make_stream({'A': float(F_A_in), 'B': 0.0}, T=350.0, P=101325.0)
        _, info = cstr(inlet, T_spec=350.0)
        conv_list.append(float(info['conversion']['A']))

# Plot conversion vs flow rate
axes[0].plot(F_A_in_range, jnp.array(conversions_true)*100, 'g-', label='True model', linewidth=2)
axes[0].plot(F_A_in_range, jnp.array(conversions_est)*100, 'r--', label='Estimated model', linewidth=2)
axes[0].scatter([d['F_A_in'] for d in measured_data], 
                [d['conversion']*100 for d in measured_data],
                s=100, c='blue', label='Measured data', zorder=5)
axes[0].set_xlabel('Inlet Flow Rate (mol/s)')
axes[0].set_ylabel('Conversion (%)')
axes[0].set_title('Conversion vs Flow Rate')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter convergence
axes[1].plot([h['k'] for h in history], 'b-', linewidth=2)
axes[1].axhline(k_true, color='g', linestyle='--', label=f'True k = {k_true}')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Rate constant k (1/s)')
axes[1].set_title('Parameter Convergence')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss surface
k_range = jnp.linspace(0.05, 0.3, 100)
def compute_loss_for_plot(k):
    total = 0.0
    for data_point in measured_data:
        cstr = CSTR(
            CSTRParams(V=V_reactor, rate_fn=rate_fn, stoich=stoich,
                      rate_params={'k': k}, species_order=['A', 'B']),
            mode="isothermal",
        )
        inlet = make_stream({'A': data_point['F_A_in'], 'B': 0.0}, T=350.0, P=101325.0)
        outlet, _ = cstr(inlet, T_spec=350.0)
        total = total + (outlet['F_A'] - data_point['F_A_out'])**2
    return total

losses = [float(compute_loss_for_plot(k)) for k in k_range]
axes[2].plot(k_range, losses, 'b-', linewidth=2)
axes[2].axvline(k_true, color='g', linestyle='--', label='True k')
axes[2].axvline(float(k_estimated), color='r', linestyle=':', label='Estimated k')
axes[2].set_xlabel('Rate constant k (1/s)')
axes[2].set_ylabel('Loss')
axes[2].set_title('Loss Surface')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Multi-Parameter Estimation

Now let's estimate multiple parameters simultaneously:
- Rate constant pre-exponential factor (A)
- Activation energy (Ea)

For Arrhenius kinetics: $k = A \cdot \exp(-E_a / RT)$

In [ ]:
# Define Arrhenius rate function
def arrhenius_rate_fn(C, T, params):
    """First-order reaction with Arrhenius kinetics."""
    A = params['A']
    Ea = params['Ea']
    R = 8.314  # J/mol/K
    k = A * jnp.exp(-Ea / (R * T))
    return jnp.array([k * C['A']])

# True parameters
A_true = 1e6  # Pre-exponential factor (1/s)
Ea_true = 50000.0  # Activation energy (J/mol)

# Generate data at different temperatures
temperatures = jnp.array([320.0, 340.0, 360.0, 380.0, 400.0])  # K
F_A_in = 1.0  # Fixed inlet flow

multi_param_data = []
key, subkey = random.split(key)

for T in temperatures:
    cstr = CSTR(
        CSTRParams(
            V=V_reactor,
            rate_fn=arrhenius_rate_fn,
            stoich=stoich,
            rate_params={'A': A_true, 'Ea': Ea_true},
            species_order=['A', 'B'],
        ),
        mode="isothermal",
    )
    
    inlet = make_stream({'A': F_A_in, 'B': 0.0}, T=float(T), P=101325.0)
    outlet, info = cstr(inlet, T_spec=float(T))
    
    # Add noise
    key, subkey = random.split(key)
    noise = random.normal(subkey) * 0.02 * F_A_in
    
    multi_param_data.append({
        'T': float(T),
        'F_A_out': float(outlet['F_A']) + noise,
        'conversion': float(info['conversion']['A']),
    })

print("Multi-temperature experimental data:")
print(f"{'T (K)':>10} {'F_A_out':>10} {'Conversion':>12}")
for d in multi_param_data:
    print(f"{d['T']:>10.1f} {d['F_A_out']:>10.3f} {d['conversion']*100:>11.1f}%")

In [ ]:
def estimate_arrhenius_params(data, params_initial, n_iterations=200):
    """Estimate A and Ea from temperature-dependent data."""
    
    def loss_fn(log_A, log_Ea):
        """Loss function (use log-scale for better optimization)."""
        A = jnp.exp(log_A)
        Ea = jnp.exp(log_Ea)
        
        total_loss = 0.0
        for d in data:
            cstr = CSTR(
                CSTRParams(
                    V=V_reactor,
                    rate_fn=arrhenius_rate_fn,
                    stoich=stoich,
                    rate_params={'A': A, 'Ea': Ea},
                    species_order=['A', 'B'],
                ),
                mode="isothermal",
            )
            inlet = make_stream({'A': F_A_in, 'B': 0.0}, T=d['T'], P=101325.0)
            outlet, _ = cstr(inlet, T_spec=d['T'])
            
            error = (outlet['F_A'] - d['F_A_out'])**2
            total_loss = total_loss + error
        
        return total_loss
    
    # Initialize in log-space
    log_A, log_Ea = jnp.log(params_initial['A']), jnp.log(params_initial['Ea'])
    
    # Adam optimizer
    m_A, v_A = 0.0, 0.0
    m_Ea, v_Ea = 0.0, 0.0
    beta1, beta2 = 0.9, 0.999
    eps = 1e-8
    lr = 0.1
    
    history = []
    
    for i in range(n_iterations):
        loss, (grad_log_A, grad_log_Ea) = value_and_grad(loss_fn, argnums=(0, 1))(log_A, log_Ea)
        
        # Adam update for log_A
        m_A = beta1 * m_A + (1 - beta1) * grad_log_A
        v_A = beta2 * v_A + (1 - beta2) * grad_log_A**2
        m_A_hat = m_A / (1 - beta1**(i+1))
        v_A_hat = v_A / (1 - beta2**(i+1))
        log_A = log_A - lr * m_A_hat / (jnp.sqrt(v_A_hat) + eps)
        
        # Adam update for log_Ea
        m_Ea = beta1 * m_Ea + (1 - beta1) * grad_log_Ea
        v_Ea = beta2 * v_Ea + (1 - beta2) * grad_log_Ea**2
        m_Ea_hat = m_Ea / (1 - beta1**(i+1))
        v_Ea_hat = v_Ea / (1 - beta2**(i+1))
        log_Ea = log_Ea - lr * m_Ea_hat / (jnp.sqrt(v_Ea_hat) + eps)
        
        A_current = jnp.exp(log_A)
        Ea_current = jnp.exp(log_Ea)
        history.append({'A': float(A_current), 'Ea': float(Ea_current), 'loss': float(loss)})
        
        if i % 40 == 0:
            print(f"Iter {i:3d}: A = {A_current:.2e}, Ea = {Ea_current:.0f} J/mol, loss = {loss:.6f}")
    
    return {'A': float(jnp.exp(log_A)), 'Ea': float(jnp.exp(log_Ea))}, history

# Run estimation
params_estimated, history = estimate_arrhenius_params(
    multi_param_data,
    params_initial={'A': 1e4, 'Ea': 40000.0},  # Initial guesses
    n_iterations=200,
)

print(f"\nTrue parameters:      A = {A_true:.2e}, Ea = {Ea_true:.0f} J/mol")
print(f"Estimated parameters: A = {params_estimated['A']:.2e}, Ea = {params_estimated['Ea']:.0f} J/mol")
print(f"A relative error: {abs(params_estimated['A'] - A_true) / A_true * 100:.2f}%")
print(f"Ea relative error: {abs(params_estimated['Ea'] - Ea_true) / Ea_true * 100:.2f}%")

In [ ]:
# Visualize multi-parameter estimation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Arrhenius plot (ln(k) vs 1/T)
R = 8.314
T_range = jnp.linspace(300, 420, 100)
k_true_arr = A_true * jnp.exp(-Ea_true / (R * T_range))
k_est_arr = params_estimated['A'] * jnp.exp(-params_estimated['Ea'] / (R * T_range))

axes[0].plot(1000/T_range, jnp.log(k_true_arr), 'g-', label='True', linewidth=2)
axes[0].plot(1000/T_range, jnp.log(k_est_arr), 'r--', label='Estimated', linewidth=2)

# Add data points
k_data = [A_true * jnp.exp(-Ea_true / (R * d['T'])) for d in multi_param_data]
axes[0].scatter(1000/jnp.array([d['T'] for d in multi_param_data]), 
                jnp.log(jnp.array(k_data)), s=100, c='blue', zorder=5)

axes[0].set_xlabel('1000/T (1/K)')
axes[0].set_ylabel('ln(k)')
axes[0].set_title('Arrhenius Plot')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Parameter trajectory
axes[1].loglog([h['A'] for h in history], [h['Ea'] for h in history], 'b-', alpha=0.7)
axes[1].scatter([history[0]['A']], [history[0]['Ea']], c='green', s=100, marker='o', label='Start', zorder=5)
axes[1].scatter([history[-1]['A']], [history[-1]['Ea']], c='red', s=100, marker='*', label='End', zorder=5)
axes[1].scatter([A_true], [Ea_true], c='black', s=150, marker='x', label='True', zorder=5)
axes[1].set_xlabel('A (1/s)')
axes[1].set_ylabel('Ea (J/mol)')
axes[1].set_title('Parameter Trajectory')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss convergence
axes[2].semilogy([h['loss'] for h in history], 'b-', linewidth=2)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_title('Loss Convergence')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Dynamic Parameter Estimation

Estimate parameters from time-series data (reactor startup transients).

In [ ]:
# Generate synthetic time-series data from a CSTR startup
k_true_dyn = 0.1  # True rate constant

# Create dynamic CSTR
dynamic_cstr = DynamicCSTR(
    volume=1.0,
    rate_fn=rate_fn,
    stoich=stoich,
    species_order=['A', 'B'],
    rate_params={'k': k_true_dyn},
)

# Simulate startup
inlet = make_stream({'A': 1.0, 'B': 0.0}, T=350.0, P=101325.0)

result_true = integrate_unit(
    dynamic_cstr,
    inputs={'inlet': inlet},
    t_span=(0.0, 100.0),
    method='RK4',
    n_steps=200,
)

# Sample at discrete times with noise
sample_indices = jnp.arange(0, 201, 10)  # Every 10 steps
t_samples = result_true.trajectory.t[sample_indices]
n_A_true_samples = result_true.trajectory.y[sample_indices, 0]
n_B_true_samples = result_true.trajectory.y[sample_indices, 1]

# Add measurement noise
key, subkey = random.split(key)
noise_A = random.normal(subkey, shape=n_A_true_samples.shape) * 0.5
key, subkey = random.split(key)
noise_B = random.normal(subkey, shape=n_B_true_samples.shape) * 0.5

n_A_measured = n_A_true_samples + noise_A
n_B_measured = n_B_true_samples + noise_B

print(f"Generated {len(t_samples)} time-series measurements")
print(f"Time range: {float(t_samples[0]):.1f} to {float(t_samples[-1]):.1f} s")

In [ ]:
def estimate_k_from_dynamics(t_data, n_A_data, n_B_data, k_initial, n_iterations=100):
    """Estimate rate constant from dynamic time-series data."""
    
    def loss_fn(k):
        """Loss: sum of squared errors over trajectory."""
        # Create CSTR with current k
        cstr = DynamicCSTR(
            volume=1.0,
            rate_fn=rate_fn,
            stoich=stoich,
            species_order=['A', 'B'],
            rate_params={'k': k},
        )
        
        # Simulate
        result = integrate_unit(
            cstr,
            inputs={'inlet': inlet},
            t_span=(0.0, 100.0),
            method='RK4',
            n_steps=200,
        )
        
        # Extract predictions at measurement times
        n_A_pred = result.trajectory.y[sample_indices, 0]
        n_B_pred = result.trajectory.y[sample_indices, 1]
        
        # Compute loss
        loss_A = jnp.sum((n_A_pred - n_A_data)**2)
        loss_B = jnp.sum((n_B_pred - n_B_data)**2)
        
        return loss_A + loss_B
    
    # Optimize
    k = k_initial
    velocity = 0.0
    lr = 0.02
    momentum = 0.9
    history = []
    
    for i in range(n_iterations):
        loss, grad_k = value_and_grad(loss_fn)(k)
        velocity = momentum * velocity - lr * grad_k
        k = k + velocity
        k = jnp.maximum(k, 0.001)
        
        history.append({'k': float(k), 'loss': float(loss)})
        
        if i % 20 == 0:
            print(f"Iter {i:3d}: k = {k:.4f}, loss = {loss:.2f}")
    
    return float(k), history

# Run dynamic estimation
k_dyn_estimated, dyn_history = estimate_k_from_dynamics(
    t_samples, n_A_measured, n_B_measured,
    k_initial=0.05,
    n_iterations=100,
)

print(f"\nTrue k: {k_true_dyn}")
print(f"Estimated k: {k_dyn_estimated:.4f}")
print(f"Relative error: {abs(k_dyn_estimated - k_true_dyn) / k_true_dyn * 100:.2f}%")

In [ ]:
# Compare true and estimated trajectories
cstr_estimated = DynamicCSTR(
    volume=1.0,
    rate_fn=rate_fn,
    stoich=stoich,
    species_order=['A', 'B'],
    rate_params={'k': k_dyn_estimated},
)

result_est = integrate_unit(
    cstr_estimated,
    inputs={'inlet': inlet},
    t_span=(0.0, 100.0),
    method='RK4',
    n_steps=200,
)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Species A trajectory
axes[0].plot(result_true.trajectory.t, result_true.trajectory.y[:, 0], 'g-', 
             label='True', linewidth=2)
axes[0].plot(result_est.trajectory.t, result_est.trajectory.y[:, 0], 'r--', 
             label='Estimated', linewidth=2)
axes[0].scatter(t_samples, n_A_measured, c='blue', s=30, alpha=0.7, label='Measured')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('n_A (mol)')
axes[0].set_title('Species A Holdup')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Species B trajectory
axes[1].plot(result_true.trajectory.t, result_true.trajectory.y[:, 1], 'g-', 
             label='True', linewidth=2)
axes[1].plot(result_est.trajectory.t, result_est.trajectory.y[:, 1], 'r--', 
             label='Estimated', linewidth=2)
axes[1].scatter(t_samples, n_B_measured, c='blue', s=30, alpha=0.7, label='Measured')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('n_B (mol)')
axes[1].set_title('Species B Holdup')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Loss convergence
axes[2].semilogy([h['loss'] for h in dyn_history], 'b-', linewidth=2)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Loss')
axes[2].set_title('Dynamic Estimation Loss')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Flowsheet-Level Parameter Estimation

Estimate parameters when the unit is embedded in a flowsheet with recycles.
This demonstrates that gradients flow correctly through implicit solvers.

In [ ]:
# Create a flowsheet: Feed + Recycle → CSTR → Splitter → Product + Recycle
# Estimate the rate constant from overall conversion measurements

def create_cstr_operation(k):
    """Create a CSTR operation with given rate constant."""
    def cstr_op(inlet):
        cstr = CSTR(
            CSTRParams(
                V=1.0,
                rate_fn=rate_fn,
                stoich=stoich,
                rate_params={'k': k},
                species_order=['A', 'B'],
            ),
            mode="isothermal",
        )
        return cstr(inlet, T_spec=350.0)
    return cstr_op

def splitter(inlet, split_ratio=0.5):
    """Split stream into two: product and recycle."""
    flows = get_flows(inlet)
    
    product_flows = {s: f * (1 - split_ratio) for s, f in flows.items()}
    recycle_flows = {s: f * split_ratio for s, f in flows.items()}
    
    product = make_stream(product_flows, inlet['T'], inlet['P'])
    recycle = make_stream(recycle_flows, inlet['T'], inlet['P'])
    
    return (product, recycle), {'split_ratio': split_ratio}

def mixer(stream1, stream2):
    """Mix two streams."""
    flows1 = get_flows(stream1)
    flows2 = get_flows(stream2)
    
    mixed_flows = {s: flows1.get(s, 0.0) + flows2.get(s, 0.0) 
                   for s in set(flows1) | set(flows2)}
    
    # Average temperature weighted by flow
    F1 = sum(flows1.values())
    F2 = sum(flows2.values())
    T_mixed = (stream1['T'] * F1 + stream2['T'] * F2) / (F1 + F2 + 1e-10)
    
    return make_stream(mixed_flows, T_mixed, stream1['P']), {}

# Generate "measured" overall conversion for different feed rates
k_true_fs = 0.2
split_ratio = 0.3  # 30% recycle

feed_rates = [0.5, 1.0, 1.5, 2.0]
flowsheet_data = []

for F_feed in feed_rates:
    # Build and solve flowsheet
    fs = Flowsheet(species_order=['A', 'B'])
    
    feed = make_stream({'A': F_feed, 'B': 0.0}, T=350.0, P=101325.0)
    fs.add_feed('feed', feed)
    
    fs.add_unit(Unit(
        name='mixer',
        operation=mixer,
        inlet_names=['feed', 'recycle'],
        outlet_names=['mixed'],
    ))
    
    fs.add_unit(Unit(
        name='reactor',
        operation=create_cstr_operation(k_true_fs),
        inlet_names=['mixed'],
        outlet_names=['reactor_out'],
    ))
    
    fs.add_unit(Unit(
        name='splitter',
        operation=splitter,
        inlet_names=['reactor_out'],
        outlet_names=['product', 'recycle'],
        params={'split_ratio': split_ratio},
    ))
    
    fs.add_recycle('recycle', 'recycle')
    
    # Solve with recycle
    streams = fs.solve(tol=1e-8, max_iter=100)
    
    # Overall conversion
    F_A_feed = feed['F_A']
    F_A_product = streams['product']['F_A']
    conversion = float((F_A_feed - F_A_product) / F_A_feed)
    
    # Add noise
    key, subkey = random.split(key)
    noise = random.normal(subkey) * 0.02
    
    flowsheet_data.append({
        'F_feed': F_feed,
        'conversion': conversion + float(noise),
    })

print("Flowsheet experimental data (with recycle):")
print(f"{'F_feed':>10} {'Conversion':>12}")
for d in flowsheet_data:
    print(f"{d['F_feed']:>10.2f} {d['conversion']*100:>11.1f}%")

In [ ]:
def estimate_k_from_flowsheet(data, k_initial, n_iterations=80):
    """Estimate k from flowsheet with recycle."""
    
    def loss_fn(k):
        total_loss = 0.0
        
        for d in data:
            fs = Flowsheet(species_order=['A', 'B'])
            feed = make_stream({'A': d['F_feed'], 'B': 0.0}, T=350.0, P=101325.0)
            fs.add_feed('feed', feed)
            
            fs.add_unit(Unit(
                name='mixer',
                operation=mixer,
                inlet_names=['feed', 'recycle'],
                outlet_names=['mixed'],
            ))
            
            fs.add_unit(Unit(
                name='reactor',
                operation=create_cstr_operation(k),
                inlet_names=['mixed'],
                outlet_names=['reactor_out'],
            ))
            
            fs.add_unit(Unit(
                name='splitter',
                operation=splitter,
                inlet_names=['reactor_out'],
                outlet_names=['product', 'recycle'],
                params={'split_ratio': split_ratio},
            ))
            
            fs.add_recycle('recycle', 'recycle')
            streams = fs.solve(tol=1e-8)
            
            F_A_product = streams['product']['F_A']
            conversion_pred = (d['F_feed'] - F_A_product) / d['F_feed']
            
            error = (conversion_pred - d['conversion'])**2
            total_loss = total_loss + error
        
        return total_loss
    
    k = k_initial
    lr = 1.0
    history = []
    
    for i in range(n_iterations):
        loss, grad_k = value_and_grad(loss_fn)(k)
        k = k - lr * grad_k
        k = jnp.maximum(k, 0.01)
        
        history.append({'k': float(k), 'loss': float(loss)})
        
        # Adaptive learning rate
        if i > 0 and history[-1]['loss'] > history[-2]['loss']:
            lr *= 0.5
        
        if i % 20 == 0:
            print(f"Iter {i:3d}: k = {k:.4f}, loss = {loss:.6f}")
    
    return float(k), history

# Run estimation
k_fs_estimated, fs_history = estimate_k_from_flowsheet(
    flowsheet_data,
    k_initial=0.1,
    n_iterations=80,
)

print(f"\nTrue k: {k_true_fs}")
print(f"Estimated k: {k_fs_estimated:.4f}")
print(f"Relative error: {abs(k_fs_estimated - k_true_fs) / k_true_fs * 100:.2f}%")

## 6. Uncertainty Quantification

Beyond point estimates, we often want to know the uncertainty in our parameters.

### 6.1 Confidence Intervals via Hessian (Laplace Approximation)

At the optimum, the curvature of the loss function tells us about parameter uncertainty:

$$\text{Cov}(\theta) \approx \sigma^2 \cdot H^{-1}$$

where $H$ is the Hessian of the loss and $\sigma^2$ is the measurement variance.

In [ ]:
def compute_confidence_intervals(loss_fn, param_estimate, n_data, alpha=0.05):
    """Compute confidence intervals using the Hessian."""
    from scipy import stats
    
    # Compute Hessian at the optimum
    H = hessian(loss_fn)(param_estimate)
    
    # Estimate measurement variance from residuals
    loss_at_opt = loss_fn(param_estimate)
    sigma2 = loss_at_opt / (n_data - 1)  # Estimated variance
    
    # Parameter variance
    if jnp.ndim(H) == 0:  # Scalar case
        var_param = sigma2 / H
        std_param = jnp.sqrt(var_param)
    else:
        var_param = sigma2 * jnp.linalg.inv(H)
        std_param = jnp.sqrt(jnp.diag(var_param))
    
    # t-statistic for confidence interval
    t_val = stats.t.ppf(1 - alpha/2, n_data - 1)
    
    return {
        'estimate': float(param_estimate),
        'std': float(std_param),
        'ci_lower': float(param_estimate - t_val * std_param),
        'ci_upper': float(param_estimate + t_val * std_param),
    }

# Define loss function for the steady-state CSTR estimation
def loss_for_uncertainty(k):
    total_loss = 0.0
    for data_point in measured_data:
        cstr = CSTR(
            CSTRParams(
                V=V_reactor,
                rate_fn=rate_fn,
                stoich=stoich,
                rate_params={'k': k},
                species_order=['A', 'B'],
            ),
            mode="isothermal",
        )
        inlet = make_stream({'A': data_point['F_A_in'], 'B': 0.0}, T=350.0, P=101325.0)
        outlet, _ = cstr(inlet, T_spec=350.0)
        total_loss = total_loss + (outlet['F_A'] - data_point['F_A_out'])**2
    return total_loss

# Compute confidence interval
ci_result = compute_confidence_intervals(
    loss_for_uncertainty, 
    k_estimated, 
    n_data=len(measured_data),
)

print("Parameter Uncertainty (95% CI):")
print(f"  Estimate: k = {ci_result['estimate']:.4f}")
print(f"  Std. dev: σ = {ci_result['std']:.4f}")
print(f"  95% CI: [{ci_result['ci_lower']:.4f}, {ci_result['ci_upper']:.4f}]")
print(f"  True k = {k_true} {'✓ in CI' if ci_result['ci_lower'] <= k_true <= ci_result['ci_upper'] else '✗ not in CI'}")

### 6.2 Bayesian Parameter Estimation with MCMC

For more robust uncertainty quantification, we can use Markov Chain Monte Carlo (MCMC)
to sample from the posterior distribution of parameters.

In [ ]:
def simple_mcmc(log_posterior_fn, initial_params, n_samples=2000, step_size=0.01):
    """Simple Metropolis-Hastings MCMC sampler."""
    key = random.PRNGKey(123)
    
    samples = []
    current_params = initial_params
    current_log_prob = log_posterior_fn(current_params)
    accepted = 0
    
    for i in range(n_samples):
        # Propose new parameters
        key, subkey = random.split(key)
        proposed_params = current_params + random.normal(subkey) * step_size
        
        # Ensure positivity
        if proposed_params <= 0:
            samples.append(float(current_params))
            continue
        
        # Compute acceptance probability
        proposed_log_prob = log_posterior_fn(proposed_params)
        log_alpha = proposed_log_prob - current_log_prob
        
        # Accept or reject
        key, subkey = random.split(key)
        if jnp.log(random.uniform(subkey)) < log_alpha:
            current_params = proposed_params
            current_log_prob = proposed_log_prob
            accepted += 1
        
        samples.append(float(current_params))
    
    print(f"Acceptance rate: {accepted / n_samples * 100:.1f}%")
    return jnp.array(samples)

# Define log-posterior (log-likelihood + log-prior)
def log_posterior(k):
    # Log-prior: uniform on (0, 1)
    if k <= 0 or k > 1:
        return -jnp.inf
    log_prior = 0.0  # Uniform
    
    # Log-likelihood: Gaussian errors
    sigma = 0.02  # Assumed measurement std
    log_lik = 0.0
    
    for data_point in measured_data:
        cstr = CSTR(
            CSTRParams(
                V=V_reactor,
                rate_fn=rate_fn,
                stoich=stoich,
                rate_params={'k': k},
                species_order=['A', 'B'],
            ),
            mode="isothermal",
        )
        inlet = make_stream({'A': data_point['F_A_in'], 'B': 0.0}, T=350.0, P=101325.0)
        outlet, _ = cstr(inlet, T_spec=350.0)
        
        residual = (outlet['F_A'] - data_point['F_A_out']) / sigma
        log_lik = log_lik - 0.5 * residual**2
    
    return log_prior + log_lik

# Run MCMC
print("Running MCMC...")
mcmc_samples = simple_mcmc(
    log_posterior,
    initial_params=k_estimated,
    n_samples=5000,
    step_size=0.005,
)

# Discard burn-in
burn_in = 1000
posterior_samples = mcmc_samples[burn_in:]

print(f"\nPosterior statistics:")
print(f"  Mean: {jnp.mean(posterior_samples):.4f}")
print(f"  Std:  {jnp.std(posterior_samples):.4f}")
print(f"  2.5%: {jnp.percentile(posterior_samples, 2.5):.4f}")
print(f"  97.5%: {jnp.percentile(posterior_samples, 97.5):.4f}")

In [ ]:
# Visualize MCMC results
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Trace plot
axes[0].plot(mcmc_samples, 'b-', alpha=0.7, linewidth=0.5)
axes[0].axhline(k_true, color='g', linestyle='--', label='True k')
axes[0].axvline(burn_in, color='r', linestyle=':', label='Burn-in')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('k')
axes[0].set_title('MCMC Trace')
axes[0].legend()

# Posterior histogram
axes[1].hist(posterior_samples, bins=50, density=True, alpha=0.7, color='blue')
axes[1].axvline(k_true, color='g', linestyle='--', linewidth=2, label='True k')
axes[1].axvline(float(jnp.mean(posterior_samples)), color='r', linestyle='-', 
                linewidth=2, label='Posterior mean')
axes[1].axvline(float(jnp.percentile(posterior_samples, 2.5)), color='orange', 
                linestyle=':', linewidth=2)
axes[1].axvline(float(jnp.percentile(posterior_samples, 97.5)), color='orange', 
                linestyle=':', linewidth=2, label='95% CI')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Density')
axes[1].set_title('Posterior Distribution')
axes[1].legend()

# Autocorrelation
lags = jnp.arange(100)
mean = jnp.mean(posterior_samples)
var = jnp.var(posterior_samples)
autocorr = jnp.array([jnp.mean((posterior_samples[:-lag if lag > 0 else len(posterior_samples)] - mean) * 
                                (posterior_samples[lag:] - mean)) / var 
                       for lag in lags])
axes[2].bar(lags, autocorr, width=1, alpha=0.7)
axes[2].axhline(0, color='k', linestyle='-')
axes[2].set_xlabel('Lag')
axes[2].set_ylabel('Autocorrelation')
axes[2].set_title('Autocorrelation Function')

plt.tight_layout()
plt.show()

### 6.3 Propagating Parameter Uncertainty to Predictions

In [ ]:
# Propagate uncertainty to conversion predictions
def predict_conversion(k, F_A_in):
    """Predict conversion for given k and flow rate."""
    cstr = CSTR(
        CSTRParams(
            V=V_reactor,
            rate_fn=rate_fn,
            stoich=stoich,
            rate_params={'k': k},
            species_order=['A', 'B'],
        ),
        mode="isothermal",
    )
    inlet = make_stream({'A': F_A_in, 'B': 0.0}, T=350.0, P=101325.0)
    _, info = cstr(inlet, T_spec=350.0)
    return float(info['conversion']['A'])

# Predict at a new condition
F_A_new = 2.5

# Sample predictions from posterior
n_pred_samples = 500
sample_indices = random.choice(random.PRNGKey(456), len(posterior_samples), (n_pred_samples,))
k_samples = posterior_samples[sample_indices]

conversion_samples = jnp.array([predict_conversion(float(k), F_A_new) for k in k_samples])

# True conversion
conversion_true = predict_conversion(k_true, F_A_new)

print(f"Prediction at F_A = {F_A_new} mol/s:")
print(f"  True conversion: {conversion_true*100:.1f}%")
print(f"  Predicted mean:  {jnp.mean(conversion_samples)*100:.1f}%")
print(f"  Predicted std:   {jnp.std(conversion_samples)*100:.1f}%")
print(f"  95% CI: [{jnp.percentile(conversion_samples, 2.5)*100:.1f}%, "
      f"{jnp.percentile(conversion_samples, 97.5)*100:.1f}%]")

In [ ]:
# Plot prediction uncertainty bands
F_range = jnp.linspace(0.3, 4.0, 30)

# Get predictions for each k sample
predictions = []
for k in k_samples[:100]:  # Use subset for speed
    preds = [predict_conversion(float(k), float(F)) for F in F_range]
    predictions.append(preds)
predictions = jnp.array(predictions)

# Compute percentiles
pred_mean = jnp.mean(predictions, axis=0)
pred_lower = jnp.percentile(predictions, 2.5, axis=0)
pred_upper = jnp.percentile(predictions, 97.5, axis=0)

# True predictions
pred_true = jnp.array([predict_conversion(k_true, float(F)) for F in F_range])

plt.figure(figsize=(10, 6))
plt.fill_between(F_range, pred_lower*100, pred_upper*100, alpha=0.3, 
                  color='blue', label='95% prediction interval')
plt.plot(F_range, pred_mean*100, 'b-', linewidth=2, label='Mean prediction')
plt.plot(F_range, pred_true*100, 'g--', linewidth=2, label='True model')
plt.scatter([d['F_A_in'] for d in measured_data], 
            [d['conversion']*100 for d in measured_data],
            s=100, c='red', zorder=5, label='Measured data')
plt.xlabel('Inlet Flow Rate (mol/s)', fontsize=12)
plt.ylabel('Conversion (%)', fontsize=12)
plt.title('Model Predictions with Uncertainty', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated comprehensive parameter estimation techniques with difflow:

### Key Methods

1. **Basic gradient descent**: Simple and effective for smooth loss surfaces
   - Use `jax.value_and_grad()` to get loss and gradient simultaneously
   - Adam optimizer often works better than vanilla SGD

2. **Steady-state estimation**: Fit kinetic parameters from CSTR outlet measurements
   - Works with implicit solvers (fixed-point iteration)
   - Gradients flow through converged solutions

3. **Multi-parameter estimation**: Estimate A and Ea from temperature-dependent data
   - Use log-scale for parameters spanning orders of magnitude
   - Arrhenius plot helps visualize fit quality

4. **Dynamic estimation**: Fit parameters to time-series startup data
   - Gradients propagate through ODE integration
   - Can fit rate constants, transport coefficients, etc.

5. **Flowsheet-level estimation**: Estimate parameters in complex systems with recycles
   - Implicit differentiation through recycle convergence
   - Overall conversion measurements can identify kinetic parameters

6. **Uncertainty quantification**:
   - Hessian-based confidence intervals (quick approximation)
   - MCMC for full posterior distribution
   - Propagate uncertainty to predictions

### Best Practices

- **Normalization**: Scale parameters and loss components for balanced optimization
- **Initialization**: Start from physically reasonable initial guesses
- **Regularization**: Add priors or constraints to prevent overfitting
- **Validation**: Hold out data for testing generalization
- **Identifiability**: Check that parameters are uniquely determined by data